# `kalshi_trades` quick start

This notebook shows a minimal workflow for exploring Kalshi data with `kalshi_trades`.

You will:

- create a `Config` and `KalshiClient`
- fetch small samples of markets, events, and series
- inspect one example of each resource in more detail
- optionally subscribe to live updates for a sampled market

## Notebook structure

1. **Setup**: create the client
2. **Browse sample data**: list a few markets, events, and series
3. **Inspect details**: automatically pick tickers from the sample data
4. **Optional streaming**: subscribe to live ticker and trade updates

If your package is in this repo but not installed into the current environment yet, install it first from the terminal with `uv pip install -e .` or the equivalent command for your setup.

## 1. Set up the client

Create a `Config` and `KalshiClient` for the current environment. The remaining cells reuse this client for all API calls.

In [2]:
from pathlib import Path

import requests

from kalshi_trades import Config, KalshiClient

cfg = Config(
    env="prod",
    private_key_path=Path.cwd().parent / "kalshi_private_key.pem",
)

try:
    client = KalshiClient(cfg)
    client_mode = "authenticated"
except FileNotFoundError as err:
    # Public market/event/series endpoints do not require auth.
    # Fall back to a public-only client when the signing key is unavailable.
    client = KalshiClient.__new__(KalshiClient)
    client._config = cfg
    client._auth = None
    client._base = cfg.rest_base
    client._timeout = 10.0
    client._session = requests.Session()
    client_mode = f"public-only (missing private key file: {err.filename})"

{
    "client": client,
    "mode": client_mode,
    "rest_base": cfg.rest_base,
}

{'client': <kalshi_trades.client.KalshiClient at 0x121a3c050>,
 'mode': 'authenticated',
 'rest_base': 'https://api.elections.kalshi.com/trade-api/v2'}

## 2. Browse sample markets, events, and series

Fetch a small sample of open markets plus related events and series.

This section also stores one example object from each response so later cells can reuse them without repeating extraction logic.

In [3]:
from IPython.display import Code
import json

# Fetch small response samples for exploration.
markets = client.get_markets(status="open", limit=10)
events = client.get_events(limit=10)
series = client.get_series_list(limit=10)

# Keep one representative object from each response for downstream cells.
sample_market = markets.get("markets", [{}])[0] if markets.get("markets") else {}
sample_event = events.get("events", [{}])[0] if events.get("events") else {}
sample_series = series.get("series", [{}])[0] if series.get("series") else {}

result = {
    "market_count": len(markets.get("markets", [])),
    "event_count": len(events.get("events", [])),
    "series_count": len(series.get("series", [])),
    "sample_market": sample_market,
    "sample_event": sample_event,
    "sample_series": sample_series,
}

Code(json.dumps(result, indent=2, ensure_ascii=False), language="json")

{
  "market_count": 10,
  "event_count": 10,
  "series_count": 9436,
  "sample_market": {
    "can_close_early": true,
    "close_time": "2026-04-18T23:00:00Z",
    "created_time": "2026-04-05T00:06:33.807392Z",
    "custom_strike": {
      "Associated Events": "KXNBASPREAD-26APR04DETPHI,KXNCAAMBSPREAD-26APR04ILLCONN,KXNCAAMBSPREAD-26APR04MICHARIZ,KXUFCFIGHT-26APR04JANRIC,KXUFCFIGHT-26APR04MCMZEC,KXUFCFIGHT-26APR04MOIDUN,KXUFCFIGHT-26APR04YAKRIB",
      "Associated Market Sides": "yes,yes,no,yes,yes,yes,yes",
      "Associated Markets": "KXNBASPREAD-26APR04DETPHI-DET4,KXNCAAMBSPREAD-26APR04ILLCONN-CONN1,KXNCAAMBSPREAD-26APR04MICHARIZ-ARIZ5,KXUFCFIGHT-26APR04JANRIC-RIC,KXUFCFIGHT-26APR04MCMZEC-MCM,KXUFCFIGHT-26APR04MOIDUN-DUN,KXUFCFIGHT-26APR04YAKRIB-YAK",
      "Multivariate Event Ticker": "KXMVECROSSCATEGORY-S202650FAE2207A5"
    },
    "event_ticker": "KXMVECROSSCATEGORY-S202650FAE2207A5",
    "expected_expiration_time": "2026-04-05T05:00:00Z",
    "expiration_time": "2026-04-18T23:00:00Z",
    "expiration_value": "",
    "fractional_trading_enabled": false,
    "is_provisional": true,
    "last_price_dollars": "0.0000",
    "latest_expiration_time": "2026-04-18T23:00:00Z",
    "liquidity_dollars": "0.0000",
    "market_type": "binary",
    "mve_collection_ticker": "KXMVECROSSCATEGORY-R",
    "mve_selected_legs": [
      {
        "event_ticker": "KXNBASPREAD-26APR04DETPHI",
        "market_ticker": "KXNBASPREAD-26APR04DETPHI-DET4",
        "side": "yes"
      },
      {
        "event_ticker": "KXNCAAMBSPREAD-26APR04ILLCONN",
        "market_ticker": "KXNCAAMBSPREAD-26APR04ILLCONN-CONN1",
        "side": "yes"
      },
      {
        "event_ticker": "KXNCAAMBSPREAD-26APR04MICHARIZ",
        "market_ticker": "KXNCAAMBSPREAD-26APR04MICHARIZ-ARIZ5",
        "side": "no"
      },
      {
        "event_ticker": "KXUFCFIGHT-26APR04JANRIC",
        "market_ticker": "KXUFCFIGHT-26APR04JANRIC-RIC",
        "side": "yes"
      },
      {
        "event_ticker": "KXUFCFIGHT-26APR04MCMZEC",
        "market_ticker": "KXUFCFIGHT-26APR04MCMZEC-MCM",
        "side": "yes"
      },
      {
        "event_ticker": "KXUFCFIGHT-26APR04MOIDUN",
        "market_ticker": "KXUFCFIGHT-26APR04MOIDUN-DUN",
        "side": "yes"
      },
      {
        "event_ticker": "KXUFCFIGHT-26APR04YAKRIB",
        "market_ticker": "KXUFCFIGHT-26APR04YAKRIB-YAK",
        "side": "yes"
      }
    ],
    "no_ask_dollars": "1.0000",
    "no_bid_dollars": "1.0000",
    "no_sub_title": "yes Detroit wins by over 4.5 Points,yes UConn wins by over 1.5 Points,no Arizona wins by over 5.5 Points,yes Tabatha Ricci,yes Tommy McMillen,yes Chris Duncan,yes Abdul-Rakhman Yakhyaev",
    "notional_value_dollars": "1.0000",
    "open_interest_fp": "0.00",
    "open_time": "2026-04-05T00:06:33.786166Z",
    "previous_price_dollars": "0.0000",
    "previous_yes_ask_dollars": "0.0000",
    "previous_yes_bid_dollars": "0.0000",
    "price_level_structure": "deci_cent",
    "price_ranges": [
      {
        "end": "1.0000",
        "start": "0.0000",
        "step": "0.0010"
      }
    ],
    "response_price_units": "usd_cent",
    "result": "",
    "rules_primary": "",
    "rules_secondary": "",
    "settlement_timer_seconds": 5,
    "status": "active",
    "strike_type": "custom",
    "tick_size": 1,
    "ticker": "KXMVECROSSCATEGORY-S202650FAE2207A5-961C6A4C968",
    "title": "yes Detroit wins by over 4.5 Points,yes UConn wins by over 1.5 Points,no Arizona wins by over 5.5 Points,yes Tabatha Ricci,yes Tommy McMillen,yes Chris Duncan,yes Abdul-Rakhman Yakhyaev",
    "updated_time": "2026-04-05T00:06:33.972141Z",
    "volume_24h_fp": "0.00",
    "volume_fp": "0.00",
    "yes_ask_dollars": "0.0000",
    "yes_ask_size_fp": "0.00",
    "yes_bid_dollars": "0.0000",
    "yes_bid_size_fp": "0.00",
    "yes_sub_title": "yes Detroit wins by over 4.5 Points,yes UConn wins by over 1.5 Points,no Arizona wins by over 5.5 Points,yes Tabatha Ricci,yes Tommy McMillen,yes Chris Duncan,yes Abdul-R

## 3. Inspect one sampled market, event, and series

Use the sample objects from the previous section to automatically extract likely ticker fields.

This avoids manual placeholder replacement and makes the notebook easier to rerun from top to bottom.

In [4]:
def pick_ticker(item, *keys):
    """Return the first non-empty ticker-like value from a response object."""
    for key in keys:
        value = item.get(key)
        if value:
            return value
    return None

# Kalshi responses may use slightly different field names depending on endpoint.
market_ticker = pick_ticker(sample_market, "ticker", "market_ticker")
event_ticker = pick_ticker(sample_event, "event_ticker", "ticker")
series_ticker = pick_ticker(sample_series, "series_ticker", "ticker")

# Fetch detailed records only when a ticker was found in the sample payload.
{
    "market_ticker": market_ticker,
    "event_ticker": event_ticker,
    "series_ticker": series_ticker,
    "market": client.get_market(market_ticker) if market_ticker else None,
    "event": client.get_event(event_ticker, with_nested_markets=True) if event_ticker else None,
    "series": client.get_series(series_ticker) if series_ticker else None,
}

{'market_ticker': 'KXMVECROSSCATEGORY-S202650FAE2207A5-961C6A4C968',
 'event_ticker': 'KXELONMARS-99',
 'series_ticker': 'KXSSDELAY',
 'market': {'market': {'can_close_early': True,
   'close_time': '2026-04-18T23:00:00Z',
   'created_time': '2026-04-05T00:06:33.807392Z',
   'custom_strike': {'Associated Events': 'KXNBASPREAD-26APR04DETPHI,KXNCAAMBSPREAD-26APR04ILLCONN,KXNCAAMBSPREAD-26APR04MICHARIZ,KXUFCFIGHT-26APR04JANRIC,KXUFCFIGHT-26APR04MCMZEC,KXUFCFIGHT-26APR04MOIDUN,KXUFCFIGHT-26APR04YAKRIB',
    'Associated Market Sides': 'yes,yes,no,yes,yes,yes,yes',
    'Associated Markets': 'KXNBASPREAD-26APR04DETPHI-DET4,KXNCAAMBSPREAD-26APR04ILLCONN-CONN1,KXNCAAMBSPREAD-26APR04MICHARIZ-ARIZ5,KXUFCFIGHT-26APR04JANRIC-RIC,KXUFCFIGHT-26APR04MCMZEC-MCM,KXUFCFIGHT-26APR04MOIDUN-DUN,KXUFCFIGHT-26APR04YAKRIB-YAK',
    'Multivariate Event Ticker': 'KXMVECROSSCATEGORY-S202650FAE2207A5'},
   'event_ticker': 'KXMVECROSSCATEGORY-S202650FAE2207A5',
   'expected_expiration_time': '2026-04-05T05:00:00Z',

## 4. Optional: stream live updates for a sampled market

This section subscribes to live updates for the sampled market ticker discovered above.

Notes:

- Run this only after the earlier setup and sampling cells.
- The cell listens until you interrupt it manually.
- If no ticker is available, the cell raises an error instead of subscribing.

In [6]:
import asyncio

from kalshi_trades import KalshiWebSocket

messages = []
stream_market_ticker = market_ticker

def on_ticker(data):
    messages.append({"type": "ticker", "msg": data})

def on_trade(data):
    messages.append({"type": "trade", "msg": data})

async def run_stream():
    if not stream_market_ticker:
        raise ValueError("Run the sampling cell first so market_ticker is available.")

    ws = KalshiWebSocket(cfg)
    ws.on("ticker", on_ticker)
    ws.on("trade", on_trade)

    async def subscribe_on_connect(client):
        await client.subscribe(
            channels=["ticker", "trade"],
            market_ticker=stream_market_ticker,
        )

    await ws.run_forever(subscribe_on_connect=subscribe_on_connect)

# In a notebook, run with:
# await run_stream()

In [ ]:
# Examine the structure of the detailed market response
market_detail = client.get_market(market_ticker) if market_ticker else {}
market_detail.get("market", {}).keys()

dict_keys(['can_close_early', 'close_time', 'created_time', 'custom_strike', 'event_ticker', 'expected_expiration_time', 'expiration_time', 'expiration_value', 'fractional_trading_enabled', 'last_price_dollars', 'latest_expiration_time', 'liquidity_dollars', 'market_type', 'mve_collection_ticker', 'mve_selected_legs', 'no_ask_dollars', 'no_bid_dollars', 'no_sub_title', 'notional_value_dollars', 'open_interest_fp', 'open_time', 'previous_price_dollars', 'previous_yes_ask_dollars', 'previous_yes_bid_dollars', 'price_level_structure', 'price_ranges', 'response_price_units', 'result', 'rules_primary', 'rules_secondary', 'settlement_timer_seconds', 'status', 'strike_type', 'tick_size', 'ticker', 'title', 'updated_time', 'volume_24h_fp', 'volume_fp', 'yes_ask_dollars', 'yes_ask_size_fp', 'yes_bid_dollars', 'yes_bid_size_fp', 'yes_sub_title'])